In [1]:
import os
import pandas as pd
import numpy as np
import spacy
import networkx as nx
import matplotlib.pyplot as plt
from neo4j import GraphDatabase
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain, SequentialChain
from tqdm import tqdm
import json

In [2]:
df_rel = pd.read_csv("../data/processed/relations_extraction.csv")

In [3]:
df_rel.head()

,doc_id,subject,subject_type,relation,object,object_type,evidence_text,confidence,extraction_method,source
0,0,TICKET,TICKET,escalated_to,Mistie,TEAM,"You're welcome, Mistie. I apologize again for ...",0.75,ner,talkmap
1,4,TICKET,TICKET,escalated_to,Union Mobile,TEAM,"You're welcome, Lessie. Thank you for choosing...",0.75,ner,talkmap
2,5,ISSUE,ISSUE,occurred_on,the last,DATE,I completely understand. Let me see if I can h...,0.90,ner,talkmap
3,5,TICKET,TICKET,escalated_to,PIN,TEAM,I completely understand. Let me see if I can h...,0.75,ner,talkmap
4,7,TICKET,TICKET,escalated_to,Union Mobile,TEAM,You're welcome! Thank you for choosing Union M...,0.75,ner,talkmap


In [4]:
df_rel.size

980410

In [ ]:
df_rel_dedup = (
    df_rel
    .sort_values("confidence", ascending=False)
    .drop_duplicates(
        subset=["subject", "relation", "object"],
        keep="first"
    )
)

In [10]:
df_rel_dedup.size

50860

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="conversational", 
    provider="auto",
    max_new_tokens=512,
    temperature=0.0,
)
chat = ChatHuggingFace(llm=llm, verbose=True)

In [42]:
TEMPLATE_VALIDATE = """
You are a knowledge graph validation assistant.

Your task:
- Decide if the relation is correct
- If incorrect, fix it
- If meaningless, reject it
- Normalize entity names
- Normalize relation label

Return ONLY valid JSON.

Input relation:
Subject: {subject}
Relation: {relation}
Object: {object}
Evidence: {evidence}

Rules:
- Relation must be meaningful
- Subject and object must be real entities
- Prefer telecom/customer support semantics
- Use uppercase snake_case for relation names
- If invalid: "valid": false

Output JSON:
{{
  "subject": "",
  "relation": "",
  "object": "",
  "valid": true,
  "confidence": 0.0
}}
"""



In [48]:
df_llm = df_rel_dedup[(df_rel_dedup.confidence >= 0.70) & (df_rel_dedup.confidence < 0.90)].copy()
df_llm = df_llm.sample(n=500, random_state=42)
df_llm.reset_index(drop=True, inplace=True)

In [44]:
validate_prompt = PromptTemplate(
    input_variables=["subject", "relation", "object", "evidence"],
    template=TEMPLATE_VALIDATE
)


In [45]:
validate_chain = LLMChain(
    llm=chat,
    prompt=validate_prompt
)

In [46]:
def validate_relation(row):
    try:
        response = validate_chain.run(
            subject=row["subject"],
            relation=row["relation"],
            object=row["object"],
            evidence=row["evidence_text"]
        )

        parsed = json.loads(response)
        parsed["doc_id"] = row["doc_id"]
        parsed["source"] = row["source"]

        return parsed

    except Exception as e:
        return {
            "subject": row["subject"],
            "relation": row["relation"],
            "object": row["object"],
            "valid": False,
            "confidence": 0.0,
            "doc_id": row["doc_id"],
            "source": row["source"],
            "error": str(e)
        }


In [49]:
validated = []

for _, row in tqdm(df_llm.iterrows(), total=len(df_llm)):
    validated.append(validate_relation(row))


100%|██████████| 500/500 [05:40<00:00,  1.47it/s]


In [50]:
df_validated = pd.DataFrame(validated)

df_validated.head(10)


,subject,relation,object,valid,confidence,doc_id,source,error
0,TICKET,escalated_to,Bernadine,False,0.0,49072,talkmap,Expecting value: line 1 column 1 (char 0)
1,TICKET,escalated_to,Comcast Internet Billing Problems/Disrespectfu...,False,0.0,100173,comcast,Expecting value: line 1 column 1 (char 0)
2,TICKET,escalated_to,Zagg,False,0.0,83210,talkmap,Extra data: line 9 column 1 (char 131)
3,account,HAS_BILLING_AMOUNT,$20,True,1.0,82734,talkmap,NaN
4,TICKET,ESCALATED_TO_CUSTOMER_SUPPORT,CUSTOMER_SUPPORT_TEAM,True,0.0,128473,bitext,NaN
5,TICKET,escalated_to,the Union Mobile Wireless Headphones 700,False,0.0,39937,talkmap,Expecting value: line 1 column 1 (char 0)
6,TICKET,escalated_to,the Unlimited Messaging Plan,False,0.0,86019,talkmap,Expecting value: line 1 column 1 (char 0)
7,TICKET,ESCALATED_TO,CUSTOMER_SUPPORT_AGENT,True,0.8,39414,talkmap,NaN
8,ISSUE,OPERATED_BY_LOCATION,CANCUN,True,0.0,140736,bitext,NaN
9,TICKET,escalated_to,I'ime,False,0.0,93123,talkmap,Expecting value: line 1 column 1 (char 0)


In [51]:
df_valid = df_validated[df_validated["valid"] == True]

df_valid = df_valid.drop_duplicates(
    subset=["subject", "relation", "object"]
)

In [52]:
df_valid.to_csv("../data/processed/relations_validated.csv", index=False)

In [53]:
df_high_conf = df_rel_dedup[df_rel_dedup["confidence"] >= 0.90].copy()
df_high_conf["llm_valid"] = True
df_high_conf["llm_confidence"] = df_high_conf["confidence"]

df_final = pd.concat([df_high_conf, df_validated], ignore_index=True)


In [54]:
df_final["final_confidence"] = (
    df_final["confidence"].fillna(0) * 0.5 +
    df_final["llm_confidence"].fillna(0) * 0.5
)

In [55]:
df_final = df_final[df_final.llm_valid == True]
df_final = df_final.sort_values("final_confidence", ascending=False)


In [56]:
df_final = df_final.drop_duplicates(
    subset=["subject", "relation", "object"],
    keep="first"
)


In [57]:
df_final.size

17025

In [59]:
df_final.sample(10)

,doc_id,subject,subject_type,relation,object,object_type,evidence_text,confidence,extraction_method,source,llm_valid,llm_confidence,valid,error,final_confidence
940,87152,ISSUE,ISSUE,occurred_on,around an hour,TIME,Certainly. The installation process is relativ...,0.90,ner,talkmap,True,0.90,NaN,NaN,0.90
561,8850,ISSUE,ISSUE,occurred_on,one-day,DATE,"Okay, it looks like there was an issue with th...",0.90,ner,talkmap,True,0.90,NaN,NaN,0.90
294,133074,ISSUE,ISSUE,occurred_on,a few business days,DATE,I'll take care of it! I'm here to provide you ...,0.90,ner,bitext,True,0.90,NaN,NaN,0.90
453,63790,ISSUE,ISSUE,occurred_on,the 15th,DATE,"Yes, that sounds good. Can you also prorate th...",0.90,ner,talkmap,True,0.90,NaN,NaN,0.90
552,74439,ISSUE,ISSUE,occurred_on,1000 minute and,TIME,Great. I'm going to go ahead and schedule that...,0.90,ner,talkmap,True,0.90,NaN,NaN,0.90
866,100842,ISSUE,ISSUE,occurred_on,48178,DATE,"Internet Help @ , South Lyon, MI 48178",0.90,ner,comcast,True,0.90,NaN,NaN,0.90
152,84701,ISSUE,ISSUE,related_to_account,RF3456789,ACCOUNT_ID,"Sure, it's RF3456789.",0.95,regex,talkmap,True,0.95,NaN,NaN,0.95
70,1135,ISSUE,ISSUE,related_to_account,UMX1234,ACCOUNT_ID,"Sure, the model number of my phone is UMX1234,...",0.95,regex,talkmap,True,0.95,NaN,NaN,0.95
901,96846,ISSUE,ISSUE,occurred_on,3000 minutes,TIME,"Of course, we can add an additional 5GB of dat...",0.90,ner,talkmap,True,0.90,NaN,NaN,0.90
535,61516,ISSUE,ISSUE,occurred_on,Easter,DATE,"Alright, thank you for your help. Easter.",0.90,ner,talkmap,True,0.90,NaN,NaN,0.90


In [3]:
df_final=pd.read_csv("../data/processed/relations_llm_validated.csv")

In [ ]:
print("Rows with problematic subject / object:")
problematic = df_final[
    df_final["subject"].isin(["NaN", "nan", np.nan, None, ""]) |
    df_final["object"].isin(["NaN", "nan", np.nan, None, ""])
]

print(problematic[["subject", "object", "relation", "final_confidence"]])

print("\nValue counts of subject that look suspicious:")
print(df_final["subject"].value_counts().head(15))

print("\nValue counts of object that look suspicious:")
print(df_final["object"].value_counts().head(15))

print("\ndf_final.dtypes:\n", df_final[["subject","object","final_confidence"]].dtypes)

Rows with problematic subject / object:
      subject object            relation  final_confidence
122  CUSTOMER    NaN  related_to_account              0.95

Value counts of subject that look suspicious:
subject
ISSUE       1105
CUSTOMER      30
Name: count, dtype: int64

Value counts of object that look suspicious:
object
PB1000           2
5                2
XX1234567890     2
JKL87654321      2
GC1234           2
FGH4567890       2
XX12345          2
JKL4234567890    2
ESL01877347      2
JKL34567890      2
XX456789         2
GZK-1234         2
UMGG1000         2
ABC123456        2
XYZ123456        2
Name: count, dtype: int64

df_final.dtypes:
 subject              object
object               object
final_confidence    float64
dtype: object


In [5]:
invalid_values = {'NaN', 'nan', 'N/A', 'null', '', None}

df_final['subject'] = df_final['subject'].astype(str).str.strip().replace(invalid_values, pd.NA)
df_final['object']  = df_final['object'].astype(str).str.strip().replace(invalid_values, pd.NA)

df_final_clean = df_final[
    df_final['subject'].notna() & 
    df_final['object'].notna() &
    (df_final['subject'] != '') &
    (df_final['object'] != '')
].copy()

print(f"Original rows : {len(df_final):,}")
print(f"Cleaned rows  : {len(df_final_clean):,}")
print(f"Dropped       : {len(df_final) - len(df_final_clean)}")

Original rows : 1,135
Cleaned rows  : 1,134
Dropped       : 1


In [6]:
dropped = df_final[
    df_final['subject'].isna() | 
    df_final['object'].isna() |
    (df_final['subject'] == '') |
    (df_final['object'] == '')
]
if not dropped.empty:
    print("\nDropped rows preview:")
    print(dropped[['subject', 'object', 'relation', 'final_confidence']])


Dropped rows preview:
      subject object            relation  final_confidence
122  CUSTOMER   <NA>  related_to_account              0.95


In [7]:
df_final.to_csv("../data/processed/relations_llm_validated.csv", index=False)
